# Transfer learning: Feature Extraction

Transfer learning is leveraging a working model's existing architecture and learning patterns for out own problem

## Downloading and becoming one with the data

In [3]:
# Download the data
#!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

In [5]:
# Get data (10% of food classes from Food101)\
import zipfile

# Unzip the downloaded file 
zip_ref = zipfile.ZipFile("10_food_classes_10_percent.zip")
zip_ref.extractall()
zip_ref.close()

In [11]:
# How many images in each folder?
import os

#Walk through data directory and list number of files
for dirpath, dirnames, filenames in os.walk("10_food_classes_10_percent"):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in {dirpath}")

There are 2 directories and 0 images in 10_food_classes_10_percent
There are 10 directories and 0 images in 10_food_classes_10_percent/train
There are 0 directories and 75 images in 10_food_classes_10_percent/train/fried_rice
There are 0 directories and 75 images in 10_food_classes_10_percent/train/hamburger
There are 0 directories and 75 images in 10_food_classes_10_percent/train/chicken_curry
There are 0 directories and 75 images in 10_food_classes_10_percent/train/sushi
There are 0 directories and 75 images in 10_food_classes_10_percent/train/ramen
There are 0 directories and 75 images in 10_food_classes_10_percent/train/ice_cream
There are 0 directories and 75 images in 10_food_classes_10_percent/train/chicken_wings
There are 0 directories and 75 images in 10_food_classes_10_percent/train/steak
There are 0 directories and 75 images in 10_food_classes_10_percent/train/grilled_salmon
There are 0 directories and 75 images in 10_food_classes_10_percent/train/pizza
There are 10 director

### Create data loaders (preparing the data)

Use the ImageDataGenerator class to load in out images in batches

In [27]:
# Setup data inputs 
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE = (224,224)
BATCH_SIZE = 32
EPOCHS = 5

train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

print("Training data")
train_data_10_percent = train_datagen.flow_from_directory(train_dir,
                                                         target_size=IMAGE_SIZE,
                                                         batch_size=BATCH_SIZE,
                                                         class_mode="categorical")
print("Test data")
train_data_10_percent = train_datagen.flow_from_directory(test_dir,
                                                         target_size=IMAGE_SIZE,
                                                         batch_size=BATCH_SIZE,
                                                         class_mode="categorical")

Training data
Found 750 images belonging to 10 classes.
Test data
Found 2500 images belonging to 10 classes.


## Setting up callbacks (things that run whilst our model trains)

Callbacks are extra functionalities you can add to your models to be performed during or after training. Some of the most popular are:
* Tracking experiments with the Tensorboard callback
* Model checkpoint with the ModelCheckpoint callback
* Stopping a model from training(before it trains too long and overfits) with the EarlyStopping callback.

In [11]:
# Create Tensorboard callback (functionize because we need to create a new for each experiment)
import datetime
import tensorflow as tf

def create_tensorboard_callback(dir_name, experimment_name):
    log_dir = dir_name + "/" + experimment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M")
    tensorbord_callback = tf.keras.callbacks.Tensorboard(log_dir=log_dir)
    print(f"Saving Tensorflow log files to: {log_dir}")
    return tensorbord_callback

## Creating models using Tensorflow Hub

In [18]:
import tensorflow_hub as hub
from tensorflow.keras import layers

In [17]:
# Compare the following models
efficientnet_v2 = "https://www.kaggle.com/models/google/efficientnet-v2/TensorFlow2/imagenet1k-b0-classification/2"
mobilnet_v2 = "https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-128-classification/2"
resnet_v2 = "https://www.kaggle.com/models/google/resnet-v2/TensorFlow2/101-classification/2"

In [42]:
# Let's create a create_model() function to create a model from a URL
def create_model(model_url, num_classes=10):
    """
    Takes a Tensorflow Hub URL and creates a Keras Sequential model wiht it.

    Args:
        model_url(str): A Tensorflow Hub feature extraction URL.
        num_classes(int): Number of output neurons in the output layer, 
        should be equal to number of target classes.

    Return:
        An uncompiled Keras Sequential model with model_url as feature extractor
        layer and Dense output layer with num_classes output neurons.
    """
    # Download the pretrained model and save it as a Keras Layer
    feature_extractor_layer = hub.KerasLayer(model_url,
                                            trainable = False,
                                            name="feature_extraction_layer",
                                            input_shape=IMAGE_SIZE+(3,))

    # Create our own model
    model = tf.keras.Sequential([
        feature_extractor_layer,
        layers.Dense(num_classes,activation="softmax", name="output_layer")
    ])

In [43]:
# Create Resnet model
resnet_model = create_model(resnet_v2, num_classes=train_data_10_percent.num_classes)
resnet_model.summary()

ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x773498a966c0> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)

In [39]:
# Create Efficientnet model
resnet_model = create_model(efficientnet_v2, num_classes=train_data_10_percent.num_classes)
resnet_model.summary()

/home/hacktheduck/Projects/tf_course/.venv/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_3 (Lambda)               │ (None, 1000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │        10,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,010 (39.10 KB)

 Trainable params: 10,010 (39.10 KB)

 Non-trainable params: 0 (0.00 B)